# FCM para Delineamento de Zonas de Manejo Agrícolas

Pipeline de clustering **Fuzzy C-Means (FCM)** para delineamento de zonas de manejo
a partir de datacubes espacio-temporais de índices de vegetação Sentinel-2.

**Fluxo de processamento:**
1. Carregamento do datacube gerado pelo pipeline de pré-processamento
2. Extração de features espacio-temporais por pixel *(etapa intercambiável)*
3. Clustering **Fuzzy C-Means** no espaço de features
4. Avaliação da partição via **FPC** e visualização das zonas

---

### Sobre a extração de features

A etapa de extração de features (Seção 3) é **independente do método de clustering**
e pode ser substituída por qualquer abordagem que produza um vetor de features por pixel:

- **Flatten temporal-espectral** *(baseline incluído neste notebook)*: concatenação direta
  de todos os índices em todos os períodos → vetor de dimensão `T × F`
- **Redução de dimensionalidade** (PCA, UMAP): compressão do vetor baseline
- **Modelos de aprendizado de máquina**: autoencoders, redes convolucionais,
  transformers, entre outros — produzem representações mais compactas e
  potencialmente mais discriminativas da variabilidade temporal

> Este notebook utiliza o baseline como extrator padrão para fins de demonstração.
> Para substituir por um extrator treinado, basta fornecer o array `features_validas`
> de shape `(N_pixels, D)` na Seção 3 antes de prosseguir para o clustering.


## 0. Dependências


In [ ]:
# Instalar scikit-fuzzy se necessário
!pip install scikit-fuzzy --quiet


## 1. Importações e Configuração


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import skfuzzy as fuzz

# Verificar disponibilidade de GPU (opcional — FCM roda em CPU)
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    print(f'TensorFlow {tf.__version__}  |  GPUs: {len(gpus)}')
except ImportError:
    print('TensorFlow não instalado — clustering FCM não requer GPU.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURAÇÃO — ajuste os caminhos e hiperparâmetros conforme necessário
# ─────────────────────────────────────────────────────────────────────────────
CONFIG = {
    # Caminhos dos datacubes gerados pelo pipeline de pré-processamento
    'caminho_iv':       '/dados/clientes/Nome_Cliente/DATACUBE/IV_Nome_Cliente_norm01.npy',
    'caminho_estatico': '/dados/clientes/Nome_Cliente/DATACUBE/Estatico_Nome_Cliente_norm01.npy',

    # Redução de dimensionalidade (0 = sem PCA, usar features brutas)
    'n_componentes_pca': 20,

    # Clustering
    'n_zonas':       3,      # número de zonas de manejo
    'n_zonas_range': (2, 7), # intervalo para avaliação automática pelo FPC
    'fcm_m':         2.0,    # coeficiente de fuzzificação (m=2 é o padrão)
    'fcm_error':     0.005,
    'fcm_maxiter':   1000,
    'seed':          42,
}


## 2. Carregamento e Inspeção do Datacube

O datacube de índices de vegetação tem shape `(T, H, W, F)`, onde:
- **T** = períodos temporais (meses ou semanas da série)
- **H, W** = dimensões espaciais em pixels
- **F** = número de índices de vegetação (NDVI, NDRE, GNDVI, etc.)

Pixels fora da área de interesse estão codificados como `NaN`.


In [ ]:
dc_iv  = np.load(CONFIG['caminho_iv'],       allow_pickle=False)  # (T, H, W, F)
dc_est = np.load(CONFIG['caminho_estatico'], allow_pickle=False)  # (V, H, W)

T, H, W, F = dc_iv.shape
V = dc_est.shape[0]

print(f'Datacube IV      : {dc_iv.shape}  (T={T} períodos, {H}×{W} px, F={F} índices)')
print(f'Datacube Estático: {dc_est.shape} ({V} variáveis topográficas)')

# Máscara de pixels válidos (dentro da AOI em todos os períodos)
mascara_valida = ~np.any(np.isnan(dc_iv), axis=(0, 3))  # (H, W) bool
n_validos = mascara_valida.sum()
print(f'Pixels válidos   : {n_validos} de {H*W} ({n_validos/(H*W)*100:.1f}%)')

# Visualização rápida: NDVI médio ao longo do tempo
ndvi_media = np.nanmean(dc_iv[:, :, :, 0], axis=0)  # índice 0 = NDVI
plt.figure(figsize=(8, 5))
plt.imshow(ndvi_media, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(label='NDVI médio (norm.)')
plt.title('NDVI médio — série temporal completa')
plt.axis('off')
plt.tight_layout()
plt.show()


## 3. Extração de Features por Pixel

Cada pixel é representado por um vetor de features que captura
sua variabilidade temporal e espectral ao longo da série.

### Baseline: flatten temporal-espectral

A abordagem mais simples concatena todos os índices em todos os períodos,
produzindo um vetor de dimensão `T × F` por pixel.
As variáveis topográficas são opcionalmente concatenadas ao final.

> **Substituição por extrator personalizado:** se você dispõe de um modelo
> treinado (autoencoder, CNN, etc.), basta substituir o bloco abaixo
> pelo código de inferência do seu modelo e fornecer `features_validas`
> com shape `(N_pixels_válidos, D)` antes de prosseguir para a Seção 4.


In [ ]:
# ── Flatten temporal-espectral ────────────────────────────────────────────
# dc_iv shape: (T, H, W, F)  →  por pixel: (T*F,)
# Transpor para (H, W, T, F) e depois reshape para (H*W, T*F)
iv_transposto = dc_iv.transpose(1, 2, 0, 3)         # (H, W, T, F)
iv_flat       = iv_transposto.reshape(H * W, T * F)  # (H*W, T*F)

# Incluir variáveis topográficas (opcional)
# dc_est shape: (V, H, W) → por pixel: (V,)
est_flat = dc_est.transpose(1, 2, 0).reshape(H * W, V)  # (H*W, V)

# Concatenar IV + topografia
features_all = np.concatenate([iv_flat, est_flat], axis=1)  # (H*W, T*F+V)

# Selecionar apenas pixels válidos
idx_validos     = mascara_valida.ravel()               # (H*W,) bool
features_validas = features_all[idx_validos]           # (N_validos, T*F+V)

print(f'Features por pixel: {features_validas.shape[1]} dimensões '
      f'({T}×{F} IV + {V} topografia)')
print(f'Pixels válidos    : {features_validas.shape[0]}')


### Redução de dimensionalidade (opcional)

Para datasets com muitos períodos ou índices, a redução via **PCA**
pode melhorar a qualidade do clustering ao remover redundâncias e ruído.
Defina `n_componentes_pca: 0` em `CONFIG` para desabilitar.


In [ ]:
n_comp = CONFIG['n_componentes_pca']

if n_comp > 0 and n_comp < features_validas.shape[1]:
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features_validas)

    pca = PCA(n_components=n_comp, random_state=CONFIG['seed'])
    features_validas = pca.fit_transform(features_scaled).astype('float32')

    variancia_acum = pca.explained_variance_ratio_.cumsum()[-1]
    print(f'PCA: {n_comp} componentes  →  {variancia_acum*100:.1f}% da variância explicada')
else:
    scaler = StandardScaler()
    features_validas = scaler.fit_transform(features_validas).astype('float32')
    print(f'PCA desabilitado  |  features normalizadas: {features_validas.shape}')


## 4. Fuzzy C-Means (FCM)

O **Fuzzy C-Means** agrupa os pixels no espaço de features.
Ao contrário do k-means, cada pixel recebe um **grau de pertinência** (0–1)
para cada zona. A zona dominante é o argmax da pertinência.

O **FPC (Fuzzy Partition Coefficient)** mede a nitidez da partição:

$$FPC = \frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{c} u_{ki}^2 \quad \in \left(\frac{1}{c},\, 1\right]$$

Valores próximos de **1** indicam zonas bem separadas;
valores próximos de **1/c** indicam sobreposição total entre zonas.


In [ ]:
# ── Avaliar FPC para diferentes números de zonas ──────────────────────────
n_min, n_max = CONFIG['n_zonas_range']
resultados_fpc = {}

print('Avaliando FPC por número de zonas...')
for n in range(n_min, n_max):
    _, u, _, _, _, _, fpc = fuzz.cluster.cmeans(
        features_validas.T,       # FCM espera shape (D, N)
        n,
        m=CONFIG['fcm_m'],
        error=CONFIG['fcm_error'],
        maxiter=CONFIG['fcm_maxiter'],
        seed=CONFIG['seed']
    )
    resultados_fpc[n] = fpc
    print(f'  n_zonas = {n}  →  FPC = {fpc:.4f}')

# Gráfico FPC
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(resultados_fpc.keys()), list(resultados_fpc.values()),
        'o-', linewidth=2, markersize=8, color='steelblue')
ax.set_xlabel('Número de zonas')
ax.set_ylabel('FPC')
ax.set_title('Fuzzy Partition Coefficient por número de zonas')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

melhor_n = max(resultados_fpc, key=resultados_fpc.get)
print(f'\nMelhor n pelo FPC: {melhor_n}  (FPC = {resultados_fpc[melhor_n]:.4f})')


In [ ]:
# ── FCM com o número de zonas escolhido ───────────────────────────────────
n_zonas = CONFIG['n_zonas']

cntr, u, _, _, _, _, fpc = fuzz.cluster.cmeans(
    features_validas.T,
    n_zonas,
    m=CONFIG['fcm_m'],
    error=CONFIG['fcm_error'],
    maxiter=CONFIG['fcm_maxiter'],
    seed=CONFIG['seed']
)

# Zona dominante por argmax da pertinência
zona_ids = np.argmax(u, axis=0)  # (N_validos,)

# Reconstruir mapa 2D de zonas (H × W)
mapa_zonas = np.full(H * W, -1, dtype=int)
mapa_zonas[idx_validos] = zona_ids
mapa_zonas = mapa_zonas.reshape(H, W)

print(f'FCM concluído  |  n_zonas = {n_zonas}  |  FPC = {fpc:.4f}')
print()
for z in range(n_zonas):
    n_pix = (zona_ids == z).sum()
    print(f'  Zona {z+1}: {n_pix:>6} pixels  ({n_pix/len(zona_ids)*100:.1f}%)')


## 5. Visualização das Zonas de Manejo


In [ ]:
paleta     = plt.cm.get_cmap('tab10', n_zonas)
cores      = [paleta(i) for i in range(n_zonas)]
# Pixels fora da AOI → cinza claro; zonas → cores da paleta
cmap_zonas = mcolors.ListedColormap(['#e0e0e0'] + cores)

mapa_plot = mapa_zonas.astype(float)
mapa_plot[mapa_zonas == -1] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# NDVI médio como referência
im0 = axes[0].imshow(ndvi_media, cmap='RdYlGn', vmin=0, vmax=1)
axes[0].set_title('NDVI médio (série temporal)', fontsize=12)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='NDVI norm.')

# Zonas de manejo
im1 = axes[1].imshow(
    mapa_plot + 1,   # NaN → NaN; 0..n-1 → 1..n para indexar cmap
    cmap=cmap_zonas, vmin=0, vmax=n_zonas
)
axes[1].set_title(
    f'Zonas de Manejo — FCM  (n = {n_zonas},  FPC = {fpc:.3f})', fontsize=12
)
axes[1].axis('off')
cbar = plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04,
                    ticks=np.arange(1, n_zonas + 1) + 0.5)
cbar.ax.set_yticklabels([f'Zona {z+1}' for z in range(n_zonas)])

plt.tight_layout()
plt.savefig('zonas_manejo_fcm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Mapa salvo em zonas_manejo_fcm.png')


## 6. Mapas de Pertinência por Zona

O FCM atribui a cada pixel um **grau de pertinência** a cada zona (soma = 1).
Regiões com pertinência alta (> 0,7) são claramente pertencentes a uma zona;
regiões com pertinência intermediária (~0,5) representam áreas de transição.


In [ ]:
# Reconstruir mapa de pertinência para cada zona
fig, axes = plt.subplots(1, n_zonas, figsize=(5 * n_zonas, 5))
if n_zonas == 1:
    axes = [axes]

for z in range(n_zonas):
    mapa_pert = np.full(H * W, np.nan, dtype='float32')
    mapa_pert[idx_validos] = u[z, :]
    mapa_pert = mapa_pert.reshape(H, W)

    im = axes[z].imshow(mapa_pert, cmap='hot_r', vmin=0, vmax=1)
    axes[z].set_title(f'Pertinência — Zona {z+1}', fontsize=11)
    axes[z].axis('off')
    plt.colorbar(im, ax=axes[z], fraction=0.046, pad=0.04)

plt.suptitle('Grau de Pertinência por Zona (FCM)', fontsize=13)
plt.tight_layout()
plt.show()


## Próximos Passos

### Melhoria da extração de features

A qualidade das zonas depende diretamente da qualidade das features usadas no clustering.
Abordagens mais sofisticadas para a Seção 3 podem incluir:

- **Autoencoders convolucionais 3D**: aprendem representações compactas da variabilidade
  espacio-temporal, capturando padrões que o flatten simples não detecta
- **Redes LSTM ou Transformer**: exploram dependências temporais de longo prazo
- **Algoritmos geoestatísticos derivados do PCA**: utilizam adaptações do PCA conjuntamente a modelos de correlação geoespacial (ex: MULTISPATI-PCA)
- **Representações baseadas em bandas de frequência** (ex: HANTS features): usam
  coeficientes harmônicos como features ao invés dos valores brutos

Qualquer um desses extratores pode substituir o bloco da Seção 3, bastando fornecer
`features_validas` com shape `(N_pixels, D)` antes de prosseguir para o FCM.

### Exportação GeoTIFF

Para integrar o mapa de zonas em um SIG (QGIS, ArcGIS), use `rasterio` para
reprojetar o array `mapa_zonas` para o CRS do projeto e exportar como `.tif`.

### Validação das zonas

- Comparar zonas com dados de solo ou produtividade histórica
- Avaliar estabilidade interanual treinando/aplicando em anos distintos
- Combinar FPC com outros índices de validade (Xie-Beni, índice SC)
